# Notebook 00: Setup & Connection Testing

## Overview
This notebook sets up the environment and validates all connections needed for the RAG demo:

1. **Install Dependencies**: Required Python packages
2. **Download Dataset**: HuggingFace E-commerce FAQ dataset
3. **Test AWS Connection**: Verify boto3 credentials
4. **Test LLM API**: Test OpenAI-compatible LLM endpoint
5. **Test Bedrock Embeddings**: Test Titan Embeddings
6. **Environment Setup**: Configure credentials and settings

## 1. Install Dependencies

In [1]:
# Install required packages
!pip install -q boto3 datasets pandas python-dotenv matplotlib seaborn

In [2]:
# Import libraries
import boto3
import json
import os
import pandas as pd
from datasets import load_dataset
from botocore.exceptions import ClientError, NoCredentialsError
from dotenv import load_dotenv

print("[OK] All libraries imported successfully!")

[OK] All libraries imported successfully!


---

## 2. Configure AWS Credentials

**Options to set credentials:**

### Option 1: Create `.env` file (Recommended)
Create a `.env` file in the project root:
```bash
AWS_ACCESS_KEY_ID=your_access_key_here
AWS_SECRET_ACCESS_KEY=your_secret_key_here
AWS_DEFAULT_REGION=us-east-1
```

### Option 2: Set environment variables in notebook
```python
os.environ['AWS_ACCESS_KEY_ID'] = 'your_access_key'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your_secret_key'
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
```

### Option 3: Use AWS CLI configure
Run in terminal: `aws configure`

In [3]:
# Load environment variables from .env file (if exists)
load_dotenv()

# Check if credentials are set
aws_access_key = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_key = os.getenv('AWS_SECRET_ACCESS_KEY')
aws_region = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')

if aws_access_key and aws_secret_key:
    print(f"[OK] AWS credentials loaded")
    print(f"   Region: {aws_region}")
    print(f"   Access Key: {aws_access_key[:8]}...")
else:
    print("[WARNING] AWS credentials NOT found!")
    print("Please set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY")
    print("\nYou can set them here:")
    # Uncomment and fill in your credentials:
    # os.environ['AWS_ACCESS_KEY_ID'] = 'AKIA...'
    # os.environ['AWS_SECRET_ACCESS_KEY'] = 'wJalr...'
    # os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

[OK] AWS credentials loaded
   Region: us-east-1
   Access Key: AKIAQ4J5...


## 3. Test AWS Connection

Verify that boto3 can connect to AWS using your credentials.

In [4]:
def test_aws_connection():
    """Test basic AWS connection using STS."""
    try:
        # Create STS client
        sts = boto3.client('sts', region_name=aws_region)
        
        # Get caller identity
        response = sts.get_caller_identity()
        
        print("[OK] AWS Connection Successful!")
        print(f"   Account ID: {response['Account']}")
        print(f"   User ARN: {response['Arn']}")
        print(f"   User ID: {response['UserId']}")
        return True
        
    except NoCredentialsError:
        print("[FAILED] AWS credentials not found!")
        print("   Please configure your AWS credentials.")
        return False
        
    except ClientError as e:
        print(f"[FAILED] AWS connection failed: {e}")
        return False
    
    except Exception as e:
        print(f"[FAILED] Unexpected error: {e}")
        return False

# Test connection
aws_connected = test_aws_connection()

/Users/apalakkode/Library/CloudStorage/OneDrive-PayPal/Agentic RAG/ecommerce-chatbot/venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


[OK] AWS Connection Successful!
   Account ID: 060795897274
   User ARN: arn:aws:iam::060795897274:user/boto3-user
   User ID: AIDAQ4J5XYW5E7PL52W75


---

## 4. Test Bedrock Model Access

### Step 1: Check Available Models

In [5]:
def list_available_bedrock_models():
    """List available Bedrock embedding models."""
    try:
        bedrock = boto3.client('bedrock', region_name=aws_region)
        
        response = bedrock.list_foundation_models()
        
        print("[OK] Available Bedrock Models:\n")
        
        # Filter for embedding models only
        embed_models = []
        
        for model in response['modelSummaries']:
            model_id = model['modelId']
            
            if 'embed' in model_id.lower():
                embed_models.append(model_id)
        
        print(" Embedding Models:")
        for model_id in sorted(embed_models):
            print(f"   - {model_id}")
        
        return True
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'AccessDeniedException':
            print("[FAILED] Access Denied to Bedrock")
            print("   Please ensure your IAM user has Bedrock permissions.")
        else:
            print(f"[FAILED] Error listing models: {e}")
        return False
    
    except Exception as e:
        print(f"[FAILED] Unexpected error: {e}")
        return False

if aws_connected:
    list_available_bedrock_models()
else:
    print("[WARNING] Skipping - AWS not connected")

[OK] Available Bedrock Models:

 Embedding Models:
   - amazon.nova-2-multimodal-embeddings-v1:0
   - amazon.titan-embed-g1-text-02
   - amazon.titan-embed-image-v1
   - amazon.titan-embed-image-v1:0
   - amazon.titan-embed-text-v1
   - amazon.titan-embed-text-v1:2:8k
   - amazon.titan-embed-text-v2:0
   - amazon.titan-embed-text-v2:0:8k
   - cohere.embed-english-v3
   - cohere.embed-english-v3:0:512
   - cohere.embed-multilingual-v3
   - cohere.embed-multilingual-v3:0:512
   - cohere.embed-v4:0
   - twelvelabs.marengo-embed-2-7-v1:0
   - twelvelabs.marengo-embed-3-0-v1:0


### Step 2: Test LLM API (OpenAI-compatible)

In [6]:
def test_paypal_llm():
    """Test LLM API."""
    try:
        from openai import OpenAI
        
        # Get LLM config
        llm_base_url = os.getenv("LLM_BASE_URL")
        llm_api_key = os.getenv("LLM_API_KEY")
        llm_model = os.getenv("LLM_MODEL", "gpt-4o")
        
        if not llm_base_url or not llm_api_key:
            print("[FAILED] LLM credentials not found in .env file")
            return False
        
        # Create OpenAI client with PayPal endpoint
        client = OpenAI(
            base_url=llm_base_url,
            api_key=llm_api_key
        )
        
        # Test prompt
        prompt = "Say 'Hello! I am gpt-4o running via LLM.' in one sentence."
        
        print(f" Testing LLM...")
        print(f"   Model: {llm_model}")
        print(f"   Prompt: {prompt}")
        print()
        
        # Call LLM
        response = client.chat.completions.create(
            model=llm_model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=100,
            temperature=0.1
        )
        
        response_text = response.choices[0].message.content
        
        print("[OK] LLM Response:")
        print(f"   {response_text}")
        print()
        print(f"   Model used: {response.model}")
        print(f"   Input tokens: {response.usage.prompt_tokens}")
        print(f"   Output tokens: {response.usage.completion_tokens}")
        
        return True
        
    except ImportError:
        print("[FAILED] OpenAI library not installed. Run: pip install openai")
        return False
    except Exception as e:
        print(f"[FAILED] Error: {e}")
        return False

if aws_connected:
    llm_works = test_paypal_llm()
else:
    print("[WARNING] Skipping - AWS not connected")
    llm_works = False

 Testing LLM...
   Model: gpt-4o
   Prompt: Say 'Hello! I am gpt-4o running via LLM.' in one sentence.

[OK] LLM Response:
   Hello! I am GPT-4o running via LLM.

   Model used: gpt-4o-2024-08-06
   Input tokens: 27
   Output tokens: 13


### Step 3: Test Titan Embeddings

In [8]:
def test_titan_embeddings():
    """Test Amazon Titan Embeddings API."""
    try:
        bedrock_runtime = boto3.client('bedrock-runtime', region_name=aws_region)
        
        # Model ID for Titan Embeddings V2
        model_id = 'amazon.titan-embed-text-v2:0'
        
        # Test text
        test_text = "What is your return policy?"
        
        # Prepare request
        request_body = {
            "inputText": test_text
        }
        
        print(f" Testing Titan Embeddings...")
        print(f"   Model ID: {model_id}")
        print(f"   Input: {test_text}")
        print()
        
        # Invoke model
        response = bedrock_runtime.invoke_model(
            modelId=model_id,
            body=json.dumps(request_body)
        )
        
        # Parse response
        response_body = json.loads(response['body'].read())
        embedding = response_body['embedding']
        
        print("[OK] Titan Embeddings Response:")
        print(f"   Embedding dimensions: {len(embedding)}")
        print(f"   Sample values (first 5): {embedding[:5]}")
        print(f"   Embedding range: [{min(embedding):.4f}, {max(embedding):.4f}]")
        
        return True
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'AccessDeniedException':
            print("[FAILED] Access Denied to Titan Embeddings")
            print("   Please request model access in Bedrock console:")
            print("   https://console.aws.amazon.com/bedrock/home#/modelaccess")
        else:
            print(f"[FAILED] Error: {e}")
        return False
    
    except Exception as e:
        print(f"[FAILED] Unexpected error: {e}")
        return False

if aws_connected:
    titan_works = test_titan_embeddings()
else:
    print("[WARNING] Skipping - AWS not connected")
    titan_works = False

/Users/apalakkode/Library/CloudStorage/OneDrive-PayPal/Agentic RAG/ecommerce-chatbot/venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


 Testing Titan Embeddings...
   Model ID: amazon.titan-embed-text-v2:0
   Input: What is your return policy?

[OK] Titan Embeddings Response:
   Embedding dimensions: 1024
   Sample values (first 5): [-0.05517013743519783, 0.044637203216552734, 0.02924000658094883, 0.0038835026789456606, 0.00934721902012825]
   Embedding range: [-0.0938, 0.0954]


In [9]:
import boto3                                                                                                                                                                                                       
bedrock = boto3.client('bedrock', region_name='us-east-1')                                                                                                                                                         
response = bedrock.list_foundation_models()                                                                                                                                                                        
                                                                                                                                                                                                                    
print("Models with 'claude' in name:")                                                                                                                                                                             
for model in response['modelSummaries']:                                                                                                                                                                           
    if 'claude' in model['modelId'].lower():                                                                                                                                                                       
        print(f"  - {model['modelId']}")  

Models with 'claude' in name:
  - anthropic.claude-sonnet-4-20250514-v1:0
  - anthropic.claude-haiku-4-5-20251001-v1:0
  - anthropic.claude-sonnet-4-5-20250929-v1:0
  - anthropic.claude-opus-4-1-20250805-v1:0
  - anthropic.claude-opus-4-5-20251101-v1:0
  - anthropic.claude-instant-v1:2:100k
  - anthropic.claude-v2:0:18k
  - anthropic.claude-v2:0:100k
  - anthropic.claude-v2:1:18k
  - anthropic.claude-v2:1:200k
  - anthropic.claude-3-sonnet-20240229-v1:0:28k
  - anthropic.claude-3-sonnet-20240229-v1:0:200k
  - anthropic.claude-3-sonnet-20240229-v1:0
  - anthropic.claude-3-haiku-20240307-v1:0:48k
  - anthropic.claude-3-haiku-20240307-v1:0:200k
  - anthropic.claude-3-haiku-20240307-v1:0
  - anthropic.claude-3-opus-20240229-v1:0:12k
  - anthropic.claude-3-opus-20240229-v1:0:28k
  - anthropic.claude-3-opus-20240229-v1:0:200k
  - anthropic.claude-3-opus-20240229-v1:0
  - anthropic.claude-3-5-sonnet-20240620-v1:0
  - anthropic.claude-3-5-sonnet-20241022-v2:0
  - anthropic.claude-3-7-sonnet-20



## 5. Download E-commerce FAQ Dataset

Download the HuggingFace dataset and save it locally.

In [10]:
def download_ecommerce_faq():
    """Download and explore HuggingFace Ecommerce FAQ dataset."""
    try:
        print(" Downloading E-commerce FAQ dataset...")
        
        # Load dataset
        dataset = load_dataset("Andyrasika/Ecommerce_FAQ")
        
        print("\n[OK] Dataset loaded successfully!")
        print(f"\n{dataset}")
        
        # Convert to pandas for easier viewing
        df = dataset['train'].to_pandas()
        
        print(f"\n Dataset Statistics:")
        print(f"   Total Q&A pairs: {len(df)}")
        print(f"   Columns: {list(df.columns)}")
        print(f"   Average question length: {df['question'].str.len().mean():.1f} characters")
        print(f"   Average answer length: {df['answer'].str.len().mean():.1f} characters")
        
        # Save to CSV
        output_path = '../data/ecommerce_faq.csv'
        df.to_csv(output_path, index=False)
        print(f"\n Dataset saved to: {output_path}")
        
        return df
        
    except Exception as e:
        print(f"[FAILED] Error downloading dataset: {e}")
        return None

# Download dataset
faq_df = download_ecommerce_faq()


[OK] Dataset loaded successfully!

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 79
    })
})

 Dataset Statistics:
   Total Q&A pairs: 79
   Columns: ['question', 'answer']
   Average question length: 55.2 characters
   Average answer length: 163.7 characters

 Dataset saved to: ../data/ecommerce_faq.csv


### Preview Dataset

In [11]:
if faq_df is not None:
    print(" Sample Q&A Pairs:\n")
    
    # Show first 5 rows
    for idx, row in faq_df.head(5).iterrows():
        print(f"{'='*80}")
        print(f"Q{idx+1}: {row['question']}")
        print(f"A{idx+1}: {row['answer']}")
        print()

 Sample Q&A Pairs:

Q1: How can I create an account?
A1: To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.

Q2: What payment methods do you accept?
A2: We accept major credit cards, debit cards, and PayPal as payment methods for online orders.

Q3: How can I track my order?
A3: You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.

Q4: What is your return policy?
A4: Our return policy allows you to return products within 30 days of purchase for a full refund, provided they are in their original condition and packaging. Please refer to our Returns page for detailed instructions.

Q5: Can I cancel my order?
A5: You can cancel your order if it has not been shipped yet. Please contact our customer support team with your order details, and we will assist you with the cancella

In [12]:
# Display full dataframe
if faq_df is not None:
    display(faq_df.head(10))

,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...
5,How long does shipping take?,Shipping times vary depending on the destinati...
6,Do you offer international shipping?,"Yes, we offer international shipping to select..."
7,What should I do if my package is lost or dama...,If your package is lost or damaged during tran...
8,Can I change my shipping address after placing...,"If you need to change your shipping address, p..."
9,How can I contact customer support?,You can contact our customer support team by p...


## 6. Connection Summary

Summary of all connection tests.

In [13]:
print("="*80)
print(" SETUP & CONNECTION TEST SUMMARY")
print("="*80)
print()

# Check status
status = {
    "AWS Connection": "[OK] Connected" if aws_connected else "[FAILED] Failed",
    "LLM API": "[OK] Working" if llm_works else "[FAILED] Not Available",
    "Titan Embeddings": "[OK] Working" if titan_works else "[FAILED] Not Available",
    "FAQ Dataset": "[OK] Downloaded" if faq_df is not None else "[FAILED] Failed"
}

for service, state in status.items():
    print(f"{service:.<30} {state}")

print()
print("="*80)

# Overall status
all_working = all([aws_connected, llm_works, titan_works, faq_df is not None])

if all_working:
    print("\n SUCCESS! All connections working. Ready to proceed with RAG demo!")
    print("\n Next Step: Run Notebook 01 - Data Exploration & Chunking Strategies")
else:
    print("\n[WARNING] Some connections failed. Please fix the issues above before proceeding.")
    print("\n Common Issues:")
    if not aws_connected:
        print("   - AWS: Check your access keys and region")
    if not llm_works or not titan_works:
        print("   - LLM: Check LLM_BASE_URL and LLM_API_KEY in .env file")
        print("   - Bedrock: Request model access at https://console.aws.amazon.com/bedrock/home#/modelaccess")
    if faq_df is None:
        print("   - Dataset: Check internet connection")

 SETUP & CONNECTION TEST SUMMARY

AWS Connection................ [OK] Connected
LLM API....................... [OK] Working
Titan Embeddings.............. [OK] Working
FAQ Dataset................... [OK] Downloaded


 SUCCESS! All connections working. Ready to proceed with RAG demo!

 Next Step: Run Notebook 01 - Data Exploration & Chunking Strategies


---

## 7. Save Configuration (Optional)

Save validated settings for use in other notebooks.

In [ ]:
if all_working:
    config = {
        "aws_region": aws_region,
        "llm_model": os.getenv("LLM_MODEL", "gpt-4o"),
        "llm_base_url": os.getenv("LLM_BASE_URL"),
        "titan_embed_model_id": "amazon.titan-embed-text-v2:0",
        "faq_dataset_path": "../data/ecommerce_faq.csv",
        "embedding_dimensions": 1024  # Titan v2 dimensions
    }
    
    config_path = '../data/config.json'
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    
    print(f"[OK] Configuration saved to: {config_path}")
    print("\nConfig contents:")
    print(json.dumps(config, indent=2))

---

## [OK] Notebook Complete

You've successfully:
- [OK] Installed all required packages
- [OK] Configured AWS credentials
- [OK] Tested AWS connection
- [OK] Verified LLM API access (OpenAI-compatible)
- [OK] Verified Bedrock Titan Embeddings access
- [OK] Downloaded E-commerce FAQ dataset (79 Q&A pairs)
- [OK] Saved configuration for next notebooks

**Next**: Proceed to `01_data_exploration_chunking.ipynb` to explore chunking strategies!